# Fetching Scientific Papers with OpenAlex

**MRS Spring 2026 — Tutorial MT01: Deploying Agentic AI in Materials Characterization Workflows**

In notebook 03 we did RAG over a hand-curated corpus of fake abstracts. Now
we'll show how to *build* such a corpus from real, current literature —
with no API keys, no scraping, and no Sci-Hub.

[**OpenAlex**](https://openalex.org) is a free, open scholarly database that
indexes ~250 million works (papers, preprints, books, datasets) with rich
metadata: titles, authors, abstracts, DOIs, citation counts, open-access
PDF URLs, and a hierarchical *concept* taxonomy. It is the modern
replacement for the now-defunct Microsoft Academic Graph and is *the*
go-to source for programmatic literature discovery in 2026.

## Why this matters for scientific agents

Every agentic literature workflow needs a "find me recent papers about X"
primitive. With OpenAlex you can:

- Surface papers published in the last 90 days on a specific technique.
- Pull abstracts straight into a RAG index.
- Get the open-access PDF URL when one exists, so your agent can read full text.
- Do all of this for free, with no auth and no rate-limit hassles.

## What you'll build

1. The bare minimum HTTP request to fetch a single paper by DOI.
2. **Keyword search** — "find recent perovskite ferroelectric papers".
3. **Concept search** — the structured way, using OpenAlex concept IDs.
4. A **gotcha** specific to OpenAlex: abstracts come as an *inverted index*,
   not plain text. We'll reconstruct them.
5. **Pagination** for queries that return more than 25 results.
6. The **polite pool** convention (`mailto=`) — etiquette that gets you
   higher rate limits.
7. Wrap the search as a Claude **tool** so an agent can find papers on its own.

---
## Setup

OpenAlex needs no API key — just `requests`. We use Anthropic later for the
agent demo.

In [ ]:
# !pip install requests anthropic

In [ ]:
import os, getpass, requests, json
from datetime import date, timedelta

# Replace with your real email — OpenAlex uses it for the 'polite pool'
# (faster rate limits, friendly notifications if you do something abusive).
MAILTO = "your.email@institution.edu"

OPENALEX = "https://api.openalex.org" 

---
## Part 1: One DOI in, one record out

The simplest possible call: ask OpenAlex for a single paper by DOI. The
endpoint is `/works/<doi-or-openalex-id>`, and you can pass any DOI as a
URL after `/works/`.

In [ ]:
DOI = "10.1126/science.1102896"   # Novoselov et al., "Electric Field Effect in Atomically Thin Carbon Films" (graphene)

resp = requests.get(
    f"{OPENALEX}/works/https://doi.org/{DOI}",
    params={"mailto": MAILTO},
    timeout=30,
)
resp.raise_for_status()
work = resp.json()

print(f"Title:       {work['title']}")
print(f"Year:        {work['publication_year']}")
print(f"Cited by:    {work['cited_by_count']:,}")
print(f"OA status:   {work['open_access']['oa_status']}")
print(f"OA PDF URL:  {work['open_access'].get('oa_url')}")
print(f"\nFirst 3 authors:")
for a in work["authorships"][:3]:
    print(f"  - {a['author']['display_name']} ({a['author'].get('orcid')})")

**What just happened:**
- We hit a public REST endpoint with no auth.
- The response is a single JSON object describing the work.
- `cited_by_count`, `open_access.oa_url`, and `authorships` are particularly
  high-signal fields you'll use over and over.

---
## Part 2: The inverted-index abstract gotcha

Try printing `work["abstract_inverted_index"]` and you'll see something
like `{"Convolutional": [12], "neural": [13], "networks": [14, 87], ...}`.

That's because **OpenAlex does not store plain-text abstracts** — it stores
them as an *inverted index* (each word mapped to its positions in the
abstract). This is a clever workaround for some publishers' copyright
policies on plain text, but it means you have to reconstruct the abstract
yourself. The function is short:

In [ ]:
def reconstruct_abstract(inverted_index: dict | None) -> str:
    """Reconstruct abstract text from OpenAlex inverted-index format."""
    if not inverted_index:
        return ""
    positions: list[tuple[int, str]] = []
    for word, posns in inverted_index.items():
        for p in posns:
            positions.append((p, word))
    positions.sort()
    return " ".join(w for _, w in positions)


abstract = reconstruct_abstract(work.get("abstract_inverted_index"))
print(abstract[:500] + "...")

Always wrap this around `work.get("abstract_inverted_index")` —
some records have no abstract and the field is `None`. This happens for:
- Older works that pre-date OpenAlex's coverage,
- Books and editorials,
- A subset of papers from publishers that have asked OpenAlex not to index
  the abstract (some Nature, Cell, and AAAS titles fall in this bucket).

The function above handles that — empty in, empty out. Always check
`if rec["abstract"]:` before passing the abstract downstream to RAG, an
LLM, or your own classifier.

---
## Part 3: Keyword search — *"give me recent papers about X"*

The `/works` endpoint with a `search` parameter is the workhorse. Combine
it with a date filter and a sort to get *recent* hits.

In [ ]:
def search_papers(query: str, since: str = None, per_page: int = 10) -> list[dict]:
    """Search OpenAlex for recent papers matching `query`."""
    if since is None:
        since = (date.today() - timedelta(days=180)).isoformat()
    params = {
        "search":   query,
        "filter":   f"from_publication_date:{since},type:article",
        "sort":     "publication_date:desc",
        "per_page": per_page,
        "mailto":   MAILTO,
    }
    r = requests.get(f"{OPENALEX}/works", params=params, timeout=30)
    r.raise_for_status()
    return r.json()["results"]


hits = search_papers("perovskite ferroelectric thin film", per_page=5)
for w in hits:
    print(f"- [{w['publication_date']}] {w['title'][:80]}")
    print(f"    cited_by={w['cited_by_count']}  doi={(w.get('doi') or '').replace('https://doi.org/','')}")

Three filters do most of the work:

| Filter | What it does |
|---|---|
| `from_publication_date:YYYY-MM-DD` | Lower bound on publication date |
| `type:article` | Skip preprint repositories' duplicates of the same paper, datasets, books, etc. |
| `concepts.id:!<id>` | Exclude works tagged with concept `<id>` (useful for filtering out off-topic noise) |

The full filter syntax is documented at
<https://docs.openalex.org/api-entities/works/filter-works>.

---
## Part 4: Concept search — the structured way

Keyword search has a recall problem: you'll miss papers that use synonyms
("ferroelectric" vs "piezoelectric"; "perovskite" vs "ABO3"). For
high-precision surveillance, OpenAlex's hierarchical *concept* taxonomy is
much better — every work is tagged with concept IDs by an ML classifier,
and you can search by those IDs directly.

You find concept IDs by searching the `/concepts` endpoint:

In [ ]:
def find_concept(name: str) -> dict | None:
    r = requests.get(
        f"{OPENALEX}/concepts",
        params={"search": name, "per_page": 3, "mailto": MAILTO},
        timeout=30,
    )
    r.raise_for_status()
    results = r.json()["results"]
    return results[0] if results else None


for query in ["ptychography", "X-ray fluorescence", "perovskite"]:
    c = find_concept(query)
    if c:
        print(f"{query!r:25}  id={c['id'].split('/')[-1]:10}  level={c['level']}  works={c['works_count']:,}")

Now use those IDs to search by concept. The filter syntax `concepts.id:A|B|C`
means "tagged with A *or* B *or* C":

In [ ]:
def search_by_concepts(concept_ids: list[str], since: str = None,
                       per_page: int = 10) -> list[dict]:
    """Find recent papers tagged with any of the listed concept IDs."""
    if since is None:
        since = (date.today() - timedelta(days=90)).isoformat()
    concept_or = "|".join(f"https://openalex.org/{cid}" for cid in concept_ids)
    params = {
        "filter": (
            f"from_publication_date:{since},type:article,"
            f"concepts.id:{concept_or}"
        ),
        "sort":     "publication_date:desc",
        "per_page": per_page,
        "mailto":   MAILTO,
    }
    r = requests.get(f"{OPENALEX}/works", params=params, timeout=30)
    r.raise_for_status()
    return r.json()["results"]


# Example: papers tagged with ptychography (C78153596) or coherent diffraction
# imaging (C12952745) in the last 90 days.
hits = search_by_concepts(["C78153596", "C12952745"], per_page=5)
for w in hits:
    print(f"- [{w['publication_date']}] {w['title'][:80]}")

**The `mailto` parameter** at the end of every request is OpenAlex's
*polite pool*. It's not authentication — it's the convention that says
"here's how to reach me if my code does something abusive". In return you
get higher rate limits and a friendlier reception. Always include it.

---
## Part 5: Pagination — going beyond 25 results

`per_page` caps at 200. For larger result sets, OpenAlex offers two
pagination strategies:

1. **`page=N`** — easy but capped at 10,000 total results.
2. **Cursor pagination** — use `cursor=*` on the first request, then pass
   back the `meta.next_cursor` from each response until it's `None`.

Cursor is the right choice for any non-trivial harvesting job.

In [ ]:
def harvest_all(query: str, since: str, max_pages: int = 5) -> list[dict]:
    """Cursor-paginate through results. max_pages is a safety net."""
    out, cursor = [], "*"
    for _ in range(max_pages):
        r = requests.get(
            f"{OPENALEX}/works",
            params={
                "search": query,
                "filter": f"from_publication_date:{since},type:article",
                "per_page": 50,
                "cursor": cursor,
                "mailto": MAILTO,
            },
            timeout=30,
        )
        r.raise_for_status()
        data = r.json()
        out.extend(data["results"])
        cursor = data["meta"].get("next_cursor")
        if not cursor:
            break
    return out


# Be modest in a tutorial — limit to a few pages
papers = harvest_all("X-ray ptychography", since="2025-01-01", max_pages=3)
print(f"Harvested {len(papers)} papers")

---
## Part 6: Build a clean record per paper

Real downstream code (RAG, classifiers, dashboards) wants a normalized
record, not the raw 50-field OpenAlex JSON. Here's the extraction pattern
that mirrors production projects.

In [ ]:
def to_record(work: dict) -> dict:
    """Flatten an OpenAlex work into a downstream-friendly record."""
    raw_doi = (work.get("doi") or "").replace("https://doi.org/", "")
    loc = work.get("primary_location") or {}
    oa  = work.get("open_access") or {}
    return {
        "title":         work.get("title", ""),
        "authors":       [a["author"]["display_name"]
                          for a in work.get("authorships", [])],
        "publication_date": work.get("publication_date", ""),
        "year":          work.get("publication_year"),
        "journal":       (loc.get("source") or {}).get("display_name", ""),
        "doi":           raw_doi,
        "doi_url":       f"https://doi.org/{raw_doi}" if raw_doi else None,
        "openalex_id":   work.get("id"),
        "cited_by_count": work.get("cited_by_count", 0),
        "abstract":      reconstruct_abstract(work.get("abstract_inverted_index")),
        "is_open_access": oa.get("is_oa", False),
        "pdf_url":       oa.get("oa_url") or loc.get("pdf_url"),
    }


records = [to_record(w) for w in hits[:3]]
for r in records:
    print(json.dumps({k: (v[:80] + "..." if isinstance(v, str) and len(v) > 80 else v)
                      for k, v in r.items() if k != "abstract"}, indent=2))
    print(f"  abstract preview: {r['abstract'][:120]}...")
    print()

**Notice `pdf_url`**: when present, this is a *direct* link to a free,
legally-shareable PDF. Combined with a polite `requests.get`, that's
literally all you need to download the paper. (For closed-access works
it'll be `None`; in production you'd fall back to Unpaywall, the publisher
API, or institutional access.)

---
## Part 7: Hand it to an agent

Now wrap the search as a tool the LLM can call. Same pattern as notebook
04 — define the schema, dispatch in a loop. With this in place, you can
ask Claude *"find me three recent papers about X and summarise them"* and
it does the rest.

In [ ]:
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6" 

In [ ]:
SEARCH_TOOL = {
    "name": "search_recent_papers",
    "description": "Search OpenAlex for recent scientific papers matching a query. "
                   "Returns title, authors, date, journal, and abstract for the top hits.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query":     {"type": "string", "description": "Free-text search query."},
            "since":     {"type": "string",
                          "description": "ISO date (YYYY-MM-DD); only return papers after this date."},
            "max_results": {"type": "integer", "default": 5},
        },
        "required": ["query"],
    },
}

def call_search_tool(query: str, since: str = None, max_results: int = 5) -> str:
    if since is None:
        since = (date.today() - timedelta(days=365)).isoformat()
    hits = search_papers(query, since=since, per_page=max_results)
    out = []
    for w in hits:
        rec = to_record(w)
        # Trim abstract to keep the message small
        rec["abstract"] = rec["abstract"][:500]
        out.append(rec)
    return json.dumps(out)


def run_paper_agent(user_question: str, max_turns: int = 5) -> str:
    messages = [{"role": "user", "content": user_question}]
    for _ in range(max_turns):
        resp = client.messages.create(
            model=MODEL, max_tokens=1024,
            tools=[SEARCH_TOOL], messages=messages,
        )
        messages.append({"role": "assistant", "content": resp.content})
        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        if not tool_uses:
            return "".join(b.text for b in resp.content if b.type == "text")
        results = []
        for tu in tool_uses:
            print(f"  -> search_recent_papers({json.dumps(tu.input)})")
            payload = call_search_tool(**tu.input)
            results.append({"type": "tool_result", "tool_use_id": tu.id, "content": payload})
        messages.append({"role": "user", "content": results})
    return "[max turns reached]"


answer = run_paper_agent(
    "Find three recent papers (last 12 months) about X-ray ptychography on "
    "thick samples or multislice methods. For each, give me the title, the "
    "core methodological contribution in one sentence, and the DOI."
)
print("\n=== Final answer ===\n")
print(answer)

You'll see Claude:
1. Call `search_recent_papers` with a sensible query (probably *"X-ray
   ptychography multislice thick samples"*).
2. Read the abstracts that come back.
3. Pick three relevant ones and summarise them with citations.

This is the foundation of an autonomous literature-surveillance agent —
the same pattern used by tools like Elicit, Consensus, and the nightly
research pipelines we run in our own group.

---
## What you can build on top of this

| Layer | What to add |
|---|---|
| **Storage** | SQLite/Postgres table keyed by DOI; check before inserting to dedupe. |
| **PDF download** | `requests.get(rec['pdf_url'])` + filename like `{year}_{firstauthor}_{title}.pdf`. Fall back to Unpaywall when `pdf_url` is `None`. |
| **Relevance filtering** | Pass each abstract to Claude with a domain-specific rubric; drop irrelevant. |
| **RAG** | Index abstracts (and full text where you have it) into a vector store — see notebook 03 for the embedding/retrieval loop. |
| **Daily surveillance** | Run the harvest on a cron with `from_publication_date=yesterday`; email yourself the relevant hits. |
| **Citation graphs** | OpenAlex exposes `referenced_works` and `cited_by_api_url` — easy to build co-citation networks. |

The OpenAlex docs at <https://docs.openalex.org> are the definitive
reference, and they include a generous tutorial section.

## Etiquette

- **Always** include `mailto=<your-real-email>` so you land in the polite pool.
- Don't hammer it: a `time.sleep(0.1)` between requests is courteous.
- If you build something cool with it, consider sponsoring OpenAlex
  (<https://openalex.org/donate>). They run on grants and small donations,
  and the data they make freely available is a public good for science.